# PolyLLM Chairman Model — QLoRA Fine-Tuning

Fine-tunes `mistralai/Mistral-7B-Instruct-v0.3` to act as a council chairman:
given ranked responses from 3 models, synthesize a final answer.

**Requirements:**
- Runtime → Change runtime type → **T4 GPU** (free tier)
- ~2-4 hours on T4, ~45 min on A100

**Steps (follow in order):**
1. Run **Cell 1** to install packages
2. ⚠️ **Runtime → Restart session** (required after Cell 1 — do not skip)
3. Run **Cell 2 onwards** (skip Cell 1 after restart)
4. Mount Google Drive when prompted
5. Upload `train.jsonl` and `val.jsonl` from `data/dataset/`

In [ ]:
# ── Cell 1: Install packages ──────────────────────────────────────────────────
# Using Unsloth — handles all bitsandbytes/triton/CUDA compatibility automatically
# Also gives ~2x faster training than vanilla QLoRA

!pip install "unsloth[colab-new] @ git+https://github.com/unslothai/unsloth.git" -q
!pip install --no-deps "trl>=0.11.0" "peft>=0.13.0" "accelerate>=0.34.0" \
    "datasets>=3.0.0" huggingface_hub -q

print('✅ Packages installed')
print()
print('⚠️  IMPORTANT: Go to Runtime → Restart session')
print('   Then run from Cell 2 onwards (skip Cell 1).')

In [ ]:
# ── Cell 2: Check GPU ─────────────────────────────────────────────────────────
import torch
print(f'GPU:  {torch.cuda.get_device_name(0)}')
print(f'VRAM: {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB')
print(f'CUDA: {torch.version.cuda}')

In [ ]:
# ── Cell 3: Mount Google Drive ────────────────────────────────────────────────
# Checkpoints are saved here — if Colab disconnects mid-training,
# your progress is NOT lost. You can resume from the last checkpoint.
from google.colab import drive
drive.mount('/content/drive')

import os
DRIVE_DIR = '/content/drive/MyDrive/polyllm-chairman'
os.makedirs(DRIVE_DIR, exist_ok=True)
os.makedirs(f'{DRIVE_DIR}/checkpoints', exist_ok=True)
os.makedirs(f'{DRIVE_DIR}/adapter', exist_ok=True)

print(f'✅ Google Drive mounted')
print(f'   Checkpoints → {DRIVE_DIR}/checkpoints')
print(f'   Final adapter → {DRIVE_DIR}/adapter')

In [ ]:
# ── Cell 4: HuggingFace login ─────────────────────────────────────────────────
# Get your token: https://huggingface.co/settings/tokens  (role: Write)
# Accept Mistral terms: https://huggingface.co/mistralai/Mistral-7B-Instruct-v0.3
from huggingface_hub import login

HF_TOKEN    = "hf_YOUR_TOKEN_HERE"   # ← paste your HF token
HF_USERNAME = "your-hf-username"     # ← your HuggingFace username

login(token=HF_TOKEN)
print('✅ Logged in to HuggingFace')

In [ ]:
# ── Cell 5: Load dataset (upload once, reuse from Drive after that) ───────────
import os, shutil
from google.colab import files

TRAIN_PATH = f'{DRIVE_DIR}/train.jsonl'
VAL_PATH   = f'{DRIVE_DIR}/val.jsonl'

if os.path.exists(TRAIN_PATH) and os.path.exists(VAL_PATH):
    # Already on Drive — no upload needed
    print('✅ Dataset found in Google Drive — skipping upload')
    print(f'   {TRAIN_PATH}  ({os.path.getsize(TRAIN_PATH)/1e6:.1f} MB)')
    print(f'   {VAL_PATH}  ({os.path.getsize(VAL_PATH)/1e6:.1f} MB)')
else:
    # First time — upload and save to Drive so you never have to again
    print('First time setup — upload train.jsonl and val.jsonl once.')
    print('They will be saved to Google Drive for all future sessions.\n')

    print('Select train.jsonl:')
    uploaded = files.upload()
    print('Select val.jsonl:')
    uploaded.update(files.upload())

    # Copy uploaded files to Drive
    for filename in ['train.jsonl', 'val.jsonl']:
        if filename in uploaded:
            dest = f'{DRIVE_DIR}/{filename}'
            shutil.copy(filename, dest)
            print(f'   Saved to Drive: {dest}')

    print('\n✅ Dataset saved to Google Drive — future runs will load automatically')

In [ ]:
# ── Cell 6: Load and format dataset ──────────────────────────────────────────
from datasets import load_dataset

# Load directly from Google Drive (works whether just uploaded or from previous session)
dataset = load_dataset('json', data_files={
    'train':      TRAIN_PATH,
    'validation': VAL_PATH
})

print(f'Train: {len(dataset["train"])} examples')
print(f'Val:   {len(dataset["validation"])} examples')

# Mistral instruction format
def format_example(example):
    text = (
        f"<s>[INST] <<SYS>>\n{example['instruction']}\n<</SYS>>\n\n"
        f"{example['input']} [/INST] "
        f"{example['output']} </s>"
    )
    return {'text': text}

dataset = dataset.map(
    format_example,
    remove_columns=['id', 'domain', 'instruction', 'input', 'output']
)

print(f'\nSample (first 300 chars):')
print(dataset['train'][0]['text'][:300])
print('\n✅ Dataset formatted')

In [ ]:
# ── Cell 7: Load model in 4-bit with Unsloth ─────────────────────────────────
from unsloth import FastLanguageModel
import torch

BASE_MODEL  = 'mistralai/Mistral-7B-Instruct-v0.3'
MAX_SEQ_LEN = 4096  # covers 94% of examples

print(f'Loading {BASE_MODEL} in 4-bit...')

model, tokenizer = FastLanguageModel.from_pretrained(
    model_name=BASE_MODEL,
    max_seq_length=MAX_SEQ_LEN,
    dtype=None,           # auto-detect (float16 on T4)
    load_in_4bit=True,    # 4-bit quantization
    token=HF_TOKEN,
)

tokenizer.pad_token = tokenizer.eos_token
tokenizer.padding_side = 'right'

print(f'✅ Model loaded — VRAM used: {torch.cuda.memory_allocated() / 1e9:.1f} GB')

In [ ]:
# ── Cell 8: Configure LoRA with Unsloth ──────────────────────────────────────
model = FastLanguageModel.get_peft_model(
    model,
    r=16,
    lora_alpha=32,
    lora_dropout=0.05,
    target_modules=[
        'q_proj', 'k_proj', 'v_proj', 'o_proj',
        'gate_proj', 'up_proj', 'down_proj',
    ],
    bias='none',
    use_gradient_checkpointing='unsloth',  # Unsloth's optimised version (saves more VRAM)
    random_state=42,
)

model.print_trainable_parameters()
# Expected: ~85M trainable out of 7B total (~1.2%)

In [ ]:
# ── Cell 9: Training arguments ────────────────────────────────────────────────
from transformers import TrainingArguments

# Checkpoints go to Google Drive — safe if session disconnects
CHECKPOINT_DIR = f'{DRIVE_DIR}/checkpoints'

training_args = TrainingArguments(
    output_dir=CHECKPOINT_DIR,          # ← Google Drive, not Colab temp storage
    num_train_epochs=3,
    per_device_train_batch_size=1,
    per_device_eval_batch_size=1,
    gradient_accumulation_steps=16,     # effective batch = 16
    gradient_checkpointing=True,        # saves ~40% VRAM
    optim='paged_adamw_32bit',
    learning_rate=2e-4,
    lr_scheduler_type='cosine',
    warmup_ratio=0.05,
    logging_steps=10,
    eval_steps=100,
    save_steps=100,                     # save to Drive every 100 steps
    save_total_limit=3,                 # keep last 3 checkpoints
    evaluation_strategy='steps',
    load_best_model_at_end=True,
    metric_for_best_model='eval_loss',
    fp16=True,
    report_to='none',
    dataloader_pin_memory=False,
)

print('✅ Training args configured')
print(f'   Checkpoints saved to: {CHECKPOINT_DIR}')
print(f'   Effective batch size: {training_args.per_device_train_batch_size * training_args.gradient_accumulation_steps}')
print(f'   Steps per epoch:      ~{len(dataset["train"]) // (training_args.per_device_train_batch_size * training_args.gradient_accumulation_steps)}')

In [ ]:
# ── Cell 10: Train ────────────────────────────────────────────────────────────
from trl import SFTTrainer

trainer = SFTTrainer(
    model=model,
    train_dataset=dataset['train'],
    eval_dataset=dataset['validation'],
    tokenizer=tokenizer,
    args=training_args,
    dataset_text_field='text',
    max_seq_length=MAX_SEQ_LEN,
    packing=False,
)

# ── Resume from checkpoint if session was interrupted ─────────────────────────
import os, glob
existing_checkpoints = sorted(glob.glob(f'{CHECKPOINT_DIR}/checkpoint-*'))
resume_from = existing_checkpoints[-1] if existing_checkpoints else None

if resume_from:
    print(f'🔄 Resuming from checkpoint: {resume_from}')
else:
    print('🚀 Starting fresh training...')

print(f'   Examples: {len(dataset["train"])} train, {len(dataset["validation"])} val')
print(f'   Epochs:   {training_args.num_train_epochs}')
print(f'   Max seq:  {MAX_SEQ_LEN} tokens')
print()

trainer.train(resume_from_checkpoint=resume_from)
print('\n✅ Training complete!')

In [ ]:
# ── Cell 11: Plot loss curves ─────────────────────────────────────────────────
import matplotlib.pyplot as plt

log_history = trainer.state.log_history
train_loss  = [(x['step'], x['loss'])      for x in log_history if 'loss'      in x]
eval_loss   = [(x['step'], x['eval_loss']) for x in log_history if 'eval_loss' in x]

fig, ax = plt.subplots(figsize=(10, 4))
if train_loss:
    ax.plot(*zip(*train_loss), label='Train loss', color='steelblue')
if eval_loss:
    ax.plot(*zip(*eval_loss), label='Val loss', color='tomato', linestyle='--', marker='o')
ax.set_xlabel('Step')
ax.set_ylabel('Loss')
ax.set_title('Chairman Model — Training Loss')
ax.legend()
plt.tight_layout()

# Save to Drive as well
plt.savefig(f'{DRIVE_DIR}/loss_curves.png', dpi=120)
plt.show()

print('Final train loss:', train_loss[-1][1] if train_loss else 'N/A')
print('Final val loss:  ', eval_loss[-1][1]  if eval_loss  else 'N/A')
print(f'Loss curve saved to Google Drive')

In [ ]:
# ── Cell 12: Save LoRA adapter to Google Drive ────────────────────────────────
ADAPTER_DIR = f'{DRIVE_DIR}/adapter'

trainer.model.save_pretrained(ADAPTER_DIR)
tokenizer.save_pretrained(ADAPTER_DIR)

print(f'✅ LoRA adapter saved to Google Drive: {ADAPTER_DIR}')
print()

import os
for f in sorted(os.listdir(ADAPTER_DIR)):
    size_mb = os.path.getsize(f'{ADAPTER_DIR}/{f}') / 1e6
    print(f'   {f:40s} {size_mb:.1f} MB')

In [ ]:
# ── Cell 13: Push adapter to HuggingFace Hub ──────────────────────────────────
REPO_NAME = f'{HF_USERNAME}/polyllm-chairman'

trainer.model.push_to_hub(REPO_NAME, token=HF_TOKEN)
tokenizer.push_to_hub(REPO_NAME, token=HF_TOKEN)

print(f'✅ Pushed to https://huggingface.co/{REPO_NAME}')
print(f'   Use this URL in Phase 5 (backend integration)')

In [ ]:
# ── Cell 14: Quick inference test ─────────────────────────────────────────────
import torch

TEST_QUESTION  = "Explain how the internet works at a high level."
TEST_RESPONSES = """--- Response from gemma-3-27b-it (Rank #1, Score: 1.00) ---
The internet is a global network of computers connected via standardized protocols.
Data travels in packets through routers using TCP/IP. DNS translates domain names to IPs.

--- Response from llama-3.3-70b-instruct (Rank #2, Score: 2.00) ---
At its core, the internet works by breaking information into packets, routing them across
physical infrastructure (fiber optic cables, satellites), and reassembling them at the destination.

--- Response from hermes-3-llama-3.1-405b (Rank #3, Score: 3.00) ---
The internet connects billions of devices worldwide through a layered protocol stack:
physical, data link, network (IP), transport (TCP/UDP), and application layers."""

SYSTEM = (
    "You are the Chairman of an AI Council. You are given a user's question and responses "
    "from multiple AI models, ranked by quality. Your task is to synthesize the most "
    "accurate, comprehensive, and well-structured answer possible."
)

prompt = (
    f"<s>[INST] <<SYS>>\n{SYSTEM}\n<</SYS>>\n\n"
    f"Question: {TEST_QUESTION}\n\n{TEST_RESPONSES} [/INST]"
)

model.eval()
inputs = tokenizer(prompt, return_tensors='pt').to('cuda')
with torch.no_grad():
    output = model.generate(
        **inputs,
        max_new_tokens=512,
        temperature=0.7,
        do_sample=True,
        pad_token_id=tokenizer.eos_token_id,
    )

generated = tokenizer.decode(
    output[0][inputs['input_ids'].shape[1]:],
    skip_special_tokens=True
)
print('=== MODEL OUTPUT ===')
print(generated)